# L1_T1 — Data Gathering: Skin Tone / Color Classification Datasets
### Layer 1 of the Aura pipeline · Task 1 of N · v2 (archive-based, Drive-sync-safe)

**What changed from the first version of this notebook:** the original version wrote ~100,000 individual
image files directly to a Google Drive-mounted folder. Google Drive's Colab mount is a FUSE filesystem that
syncs writes to the cloud *asynchronously in the background* — it does not wait for a file to actually finish
uploading before your code moves on to the next one. Writing tens of thousands of small files is close to a
worst-case pattern for this: a session ending (timeout, disconnect, manual stop) before the background sync
catches up causes **silent, undetected data loss** — the notebook prints "Done!" for files that never actually
made it to Drive's servers. This is exactly what happened on the first run of this notebook.

**The fix used throughout this version:**
1. Every dataset is downloaded/exported to **Colab's local disk** first (`/content/aura_local/...`) — fast,
   and local disk writes aren't subject to Drive's background-sync risk.
2. Each dataset is then bundled into **one single `.tar.gz` archive**.
3. That one archive file is copied to Drive — one large file syncs far more reliably than thousands of tiny
   ones, and its final size can be verified directly.
4. At the very end, the notebook forces a full Drive sync (`flush_and_unmount` + remount) and re-verifies
   every archive's size on Drive before declaring anything safe.

**Storage location:** `My Drive/Projects/Aura/Layer1_SkinTone/` (new path — consolidating away from the
earlier `Aura_ML/...` and `Ideas/Aura/...` folders from before this fix).

**Datasets and rough sizes** (see `Aura_Training_Dataset_Research.md` for the full pros/cons writeup):

| Dataset | Method | Approx. size | Notes |
|---|---|---|---|
| FairFace | Hugging Face `datasets` library | ~1–2 GB | Clean, single API call — no manual scraping |
| Fitzpatrick17k | CSV + per-image scraping | ~0.3 GB (post-`dermaamin.com` outage — see Step 4) | `dermaamin.com`, the source for most of the raw dataset's URLs, is permanently offline; ~3,700 usable images survive from the other source |
| MST-E | Manual download from Google | ~1–3 GB (includes video) | No stable public API — you download this one by hand |


## Step 0 — Mount Drive and set up BOTH local and Drive folder structures

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ---- Drive paths: final home for ARCHIVES ONLY (plus a few small text/CSV files) ----
DRIVE_ROOT = "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone"
DRIVE_DATASETS = f"{DRIVE_ROOT}/datasets"
DRIVE_FAIRFACE_DIR = f"{DRIVE_DATASETS}/fairface"
DRIVE_FITZ_DIR = f"{DRIVE_DATASETS}/fitzpatrick17k"
DRIVE_MSTE_DIR = f"{DRIVE_DATASETS}/mst-e"
DRIVE_MSTE_INCOMING = f"{DRIVE_MSTE_DIR}/_incoming"
MANIFEST_PATH = f"{DRIVE_ROOT}/manifest.json"

# ---- Local paths: where the actual bulk downloading/writing happens ----
LOCAL_ROOT = "/content/aura_local/Layer1_SkinTone"
LOCAL_DATASETS = f"{LOCAL_ROOT}/datasets"
LOCAL_FAIRFACE_DIR = f"{LOCAL_DATASETS}/fairface"
LOCAL_FITZ_DIR = f"{LOCAL_DATASETS}/fitzpatrick17k"
LOCAL_FITZ_IMG_DIR = f"{LOCAL_FITZ_DIR}/images"

for d in [DRIVE_ROOT, DRIVE_DATASETS, DRIVE_FAIRFACE_DIR, DRIVE_FITZ_DIR, DRIVE_MSTE_DIR, DRIVE_MSTE_INCOMING,
          LOCAL_ROOT, LOCAL_DATASETS, LOCAL_FAIRFACE_DIR, LOCAL_FITZ_DIR, LOCAL_FITZ_IMG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive root (archives land here):", DRIVE_ROOT)
print("Local root (bulk work happens here):", LOCAL_ROOT)


Mounted at /content/drive
Drive root (archives land here): /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone
Local root (bulk work happens here): /content/aura_local/Layer1_SkinTone


## Step 0.5 — Check your free Drive space
Quick sanity check before archiving/uploading anything sizeable.

In [ ]:
import shutil

total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive (as mounted) — Total: {total/1e9:.1f} GB | Used: {used/1e9:.1f} GB | Free: {free/1e9:.1f} GB")

if free / 1e9 < 10:
    print("\n⚠️  Less than 10 GB free — consider clearing space before continuing.")
else:
    print("\n✅ Enough headroom to proceed.")


Drive (as mounted) — Total: 107.4 GB | Used: 50.1 GB | Free: 57.2 GB

✅ Enough headroom to proceed.


## Step 1 — Install dependencies

In [ ]:
!pip install -q datasets huggingface_hub requests tqdm pandas pillow
print("Dependencies installed.")


Dependencies installed.


## Step 2 — A reusable "archive, upload, verify" helper

This is the core of the fix. Every dataset goes through this same function once it's fully written locally:
tar it into one file, copy that one file to Drive, and confirm the byte size on Drive actually matches the
local original before trusting it.

In [ ]:
import tarfile
import time

def archive_and_upload(local_dir, archive_filename, drive_dest_dir, arcname=None):
    """
    Bundles local_dir into a single .tar.gz, copies it to Drive, and verifies the
    copy's size matches the local archive exactly. Raises if verification fails.
    Returns the final Drive path.
    """
    local_archive_path = os.path.join("/content/aura_local", archive_filename)
    arcname = arcname or os.path.basename(local_dir.rstrip("/"))

    print(f"Archiving {local_dir} ...")
    with tarfile.open(local_archive_path, "w:gz") as tar:
        tar.add(local_dir, arcname=arcname)
    local_size = os.path.getsize(local_archive_path)
    print(f"  Local archive: {local_archive_path} ({local_size/1e6:.1f} MB)")

    drive_archive_path = os.path.join(drive_dest_dir, archive_filename)
    print(f"Copying to Drive: {drive_archive_path} ...")
    shutil.copy2(local_archive_path, drive_archive_path)

    # Give the FUSE mount a moment, then verify by size -- this is the check the
    # original notebook was missing entirely.
    time.sleep(2)
    drive_size = os.path.getsize(drive_archive_path)
    if drive_size != local_size:
        raise RuntimeError(
            f"SIZE MISMATCH after copying {archive_filename}: local={local_size} bytes, "
            f"drive={drive_size} bytes. Do not trust this file yet -- re-run this cell."
        )
    print(f"  ✅ Verified on Drive: {drive_size/1e6:.1f} MB matches local exactly.")
    return drive_archive_path, drive_size


## Step 3 — Dataset A: FairFace (download local → archive → upload → verify)

FairFace is hosted on Hugging Face (`HuggingFaceM4/FairFace`) as a ready-to-load dataset. We use the `1.25`
padding config. Everything here writes to **local disk first** (`LOCAL_FAIRFACE_DIR`) — nothing touches Drive
until the archive step at the end of this section.

In [ ]:
from datasets import load_dataset

fairface = load_dataset("HuggingFaceM4/FairFace", "1.25")
print(fairface)


README.md:   0%|          | 0.00/5.89k [00:00<?, ?B/s]

1.25/train-00000-of-00004-e7151785539779(…): reconstructing file:   0%|          |  0.00B /  468MB            

1.25/train-00000-of-00004-e7151785539779(…): downloading bytes:           |  0.00B            

1.25/train-00001-of-00004-f38b58e3987f3f(…): reconstructing file:   0%|          |  0.00B /  466MB            

1.25/train-00001-of-00004-f38b58e3987f3f(…): downloading bytes:           |  0.00B            

...

1.25/validation-00000-of-00001-09e3e67bb(…): reconstructing file:   0%|          |  0.00B /  237MB            

1.25/validation-00000-of-00001-09e3e67bb(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/86744 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10954 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'age', 'gender', 'race', 'service_test'],
        num_rows: 86744
    })
    validation: Dataset({
        features: ['image', 'age', 'gender', 'race', 'service_test'],
        num_rows: 10954
    })
})


In [ ]:
import csv
from PIL import Image
from tqdm.auto import tqdm

def export_fairface_split(split_name, hf_split, out_dir, limit=None):
    """Save a FairFace split to LOCAL disk as JPEGs + a labels.csv (age, gender, race)."""
    img_dir = os.path.join(out_dir, split_name)
    os.makedirs(img_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"{split_name}_labels.csv")

    n = len(hf_split) if limit is None else min(limit, len(hf_split))
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "age", "gender", "race"])
        for i in tqdm(range(n), desc=f"Exporting FairFace/{split_name} (local disk)"):
            ex = hf_split[i]
            fname = f"{i:06d}.jpg"
            fpath = os.path.join(img_dir, fname)
            if not os.path.exists(fpath):
                ex["image"].convert("RGB").save(fpath, "JPEG", quality=92)
            writer.writerow([fname, ex["age"], ex["gender"], ex["race"]])
    print(f"Done: {n} images -> {img_dir} (LOCAL disk, not yet on Drive)")
    return csv_path

export_fairface_split("train", fairface["train"], LOCAL_FAIRFACE_DIR)
export_fairface_split("validation", fairface["validation"], LOCAL_FAIRFACE_DIR)


Exporting FairFace/train (local disk):   0%|          | 0/86744 [00:00<?, ?it/s]

Done: 86744 images -> /content/aura_local/Layer1_SkinTone/datasets/fairface/train (LOCAL disk, not yet on Drive)


Exporting FairFace/validation (local disk):   0%|          | 0/10954 [00:00<?, ?it/s]

Done: 10954 images -> /content/aura_local/Layer1_SkinTone/datasets/fairface/validation (LOCAL disk, not yet on Drive)


'/content/aura_local/Layer1_SkinTone/datasets/fairface/validation_labels.csv'

In [ ]:
# Archive FairFace (images + both label CSVs, since they live in the same local folder) and upload.
fairface_drive_path, fairface_drive_size = archive_and_upload(
    local_dir=LOCAL_FAIRFACE_DIR,
    archive_filename="fairface.tar.gz",
    drive_dest_dir=DRIVE_FAIRFACE_DIR,
    arcname="fairface",
)


Archiving /content/aura_local/Layer1_SkinTone/datasets/fairface ...
  Local archive: /content/aura_local/fairface.tar.gz (2964.1 MB)
Copying to Drive: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/fairface/fairface.tar.gz ...
  ✅ Verified on Drive: 2964.1 MB matches local exactly.


## Step 4 — Dataset B: Fitzpatrick17k (CSV + scrape, local first)

Same pattern: download annotations + scrape images to **local disk**, then archive once at the end.

**Known and unfixable:** `dermaamin.com` — the source for most of this dataset's image URLs — is permanently
offline (confirmed via direct browser test: `DNS_PROBE_FINISHED_NXDOMAIN`). Expect ~76% of the 16,577 URLs to
fail every time, regardless of retries. Historically this leaves ~3,700 usable images, which — as verified in
an earlier run of this pipeline — are actually *better* balanced toward Fitzpatrick IV–VI than the original
full dataset was, since the dead domain had disproportionately hosted lighter-skin images.

In [ ]:
!rm -rf /content/fitzpatrick17k_repo
!git clone --depth 1 https://github.com/mattgroh/fitzpatrick17k.git /content/fitzpatrick17k_repo

import pandas as pd
fitz_csv_src = "/content/fitzpatrick17k_repo/fitzpatrick17k.csv"
fitz_df = pd.read_csv(fitz_csv_src)
print(fitz_df.shape)
fitz_df.head()


Cloning into '/content/fitzpatrick17k_repo'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 23 (delta 8), reused 18 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 3.78 MiB | 17.66 MiB/s, done.
Resolving deltas: 100% (8/8), done.
(16577, 9)


,md5hash,fitzpatrick_scale,fitzpatrick_centaur,label,nine_partition_label,three_partition_label,qc,url,url_alphanum
0,5e82a45bc5d78bd24ae9202d194423f8,3,3,drug induced pigmentary changes,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicmminoc...
1,fa2911a9b13b6f8af79cb700937cc14f,1,1,photodermatoses,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicpphoto...
2,d2bac3c9e4499032ca8e9b07c7d3bc40,2,3,dermatofibroma,benign dermal,benign,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicdderma...
3,0a94359e7eaacd7178e06b2823777789,1,1,psoriasis,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicppsori...
4,a39ec3b1f22c08a421fa20535e037bba,1,1,psoriasis,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicppsori...


In [ ]:
# Copy the annotations CSV to LOCAL first
fitz_csv_local = os.path.join(LOCAL_FITZ_DIR, "fitzpatrick17k.csv")
shutil.copy(fitz_csv_src, fitz_csv_local)
print(fitz_df.columns.tolist())
assert "url" in fitz_df.columns, "Expected a 'url' column -- check if the CSV schema changed upstream."


['md5hash', 'fitzpatrick_scale', 'fitzpatrick_centaur', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum']


In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

FAILED_LOG_LOCAL = os.path.join(LOCAL_FITZ_DIR, "download_failures.csv")
HEADERS = {"User-Agent": "Mozilla/5.0 (research dataset download; contact: your-email@example.com)"}

def download_one(row):
    idx, url = row
    if not isinstance(url, str) or not url.startswith("http"):
        return (idx, url, "skipped_invalid_url")
    out_path = os.path.join(LOCAL_FITZ_IMG_DIR, f"{idx:06d}.jpg")
    if os.path.exists(out_path):
        return (idx, url, "already_downloaded")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code == 200 and len(resp.content) > 500:
            with open(out_path, "wb") as f:
                f.write(resp.content)
            return (idx, url, "ok")
        else:
            return (idx, url, f"http_{resp.status_code}")
    except Exception as e:
        return (idx, url, f"error_{type(e).__name__}")

rows = list(zip(fitz_df.index, fitz_df["url"]))
results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(download_one, r): r for r in rows}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Downloading Fitzpatrick17k (local disk)"):
        results.append(fut.result())

results_df = pd.DataFrame(results, columns=["index", "url", "status"])
results_df.to_csv(FAILED_LOG_LOCAL, index=False)

ok = (results_df["status"] == "ok").sum()
already = (results_df["status"] == "already_downloaded").sum()
failed = len(results_df) - ok - already
print(f"\nDownloaded this run: {ok} | Already had: {already} | Failed/dead links: {failed}")
print("dermaamin.com failures are expected and permanent -- see the note above this section.")



Downloaded this run: 3887 | Already had: 0 | Failed/dead links: 12690
dermaamin.com failures are expected and permanent -- see the note above this section.


In [ ]:
# Class-balance check + cleaned usable CSV, same as before -- still operates on LOCAL files
fitz_df_indexed = fitz_df.reset_index().rename(columns={"index": "idx"})

existing_indices = {
    int(f.split(".")[0]) for f in os.listdir(LOCAL_FITZ_IMG_DIR) if f.endswith(".jpg")
}
surviving_df = fitz_df_indexed[fitz_df_indexed["idx"].isin(existing_indices)].copy()

print(f"Total surviving images on local disk: {len(surviving_df)} / {len(fitz_df_indexed)}\n")

full_counts = fitz_df_indexed["fitzpatrick_scale"].value_counts().sort_index()
surv_counts = surviving_df["fitzpatrick_scale"].value_counts().sort_index()

print("Survival rate by Fitzpatrick class:")
survival_summary = {}
for fp_class in sorted(full_counts.index):
    total = int(full_counts.get(fp_class, 0))
    survived = int(surv_counts.get(fp_class, 0))
    pct = 100 * survived / total if total else 0
    survival_summary[str(fp_class)] = {"total": total, "survived": survived, "survival_pct": round(pct, 1)}
    print(f"  Fitzpatrick {fp_class}: {survived}/{total} survived ({pct:.1f}%)")

usable_df = surviving_df[surviving_df["fitzpatrick_scale"] != -1].copy()
# NOTE: image_path here is intentionally left as a relative filename, not an absolute path --
# absolute local (/content/aura_local/...) or Drive paths won't exist in a future session.
# L1_T2 will resolve this against wherever it has extracted the archive that session.
usable_df["image_filename"] = usable_df["idx"].apply(lambda i: f"{i:06d}.jpg")

usable_csv_local = os.path.join(LOCAL_FITZ_DIR, "fitzpatrick17k_usable.csv")
usable_df.to_csv(usable_csv_local, index=False)
print(f"\nUsable for training (excludes -1/undetermined): {len(usable_df)} images")
print(f"Cleaned annotations (local): {usable_csv_local}")


Total surviving images on local disk: 3887 / 16577

Survival rate by Fitzpatrick class:
  Fitzpatrick -1: 153/565 survived (27.1%)
  Fitzpatrick 1: 161/2947 survived (5.5%)
  Fitzpatrick 2: 658/4808 survived (13.7%)
  Fitzpatrick 3: 996/3308 survived (30.1%)
  Fitzpatrick 4: 1082/2781 survived (38.9%)
  Fitzpatrick 5: 636/1533 survived (41.5%)
  Fitzpatrick 6: 201/635 survived (31.7%)

Usable for training (excludes -1/undetermined): 3734 images
Cleaned annotations (local): /content/aura_local/Layer1_SkinTone/datasets/fitzpatrick17k/fitzpatrick17k_usable.csv


In [ ]:
# Archive Fitzpatrick17k (images + CSVs, all in the same local folder) and upload.
fitz_drive_path, fitz_drive_size = archive_and_upload(
    local_dir=LOCAL_FITZ_DIR,
    archive_filename="fitzpatrick17k.tar.gz",
    drive_dest_dir=DRIVE_FITZ_DIR,
    arcname="fitzpatrick17k",
)


Archiving /content/aura_local/Layer1_SkinTone/datasets/fitzpatrick17k ...
  Local archive: /content/aura_local/fitzpatrick17k.tar.gz (233.1 MB)
Copying to Drive: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/fitzpatrick17k/fitzpatrick17k.tar.gz ...
  ✅ Verified on Drive: 233.1 MB matches local exactly.


## Step 5 — Dataset C: MST-E (manual download, uploaded directly as a single archive)

MST-E has no scriptable download, so this one's different: you download it once by hand and upload the
**archive itself** (zip/tar, not its extracted contents) straight to Drive. A single-file upload doesn't have
the many-small-files sync problem this notebook is otherwise designed around, so this is safe as-is:

1. Open **https://skintone.google/mste-dataset** in a browser tab.
2. Follow their instructions to get the dataset archive.
3. Upload that archive file (don't extract it first) into this Drive folder:
   `Projects/Aura/Layer1_SkinTone/datasets/mst-e/_incoming/`
4. Run the cell below to confirm it landed and record it in the manifest.

We deliberately do **not** extract it here — extracting would recreate the many-small-files problem, this
time directly on Drive. It stays as a single archive; whichever future notebook needs the individual images
will extract it to local disk temporarily, the same way this notebook works for everything else.

In [ ]:
archive_candidates = [f for f in os.listdir(DRIVE_MSTE_INCOMING)
                       if f.lower().endswith((".zip", ".tar", ".tar.gz", ".tgz"))]

if not archive_candidates:
    print("No archive found yet in:", DRIVE_MSTE_INCOMING)
    print("Download from https://skintone.google/mste-dataset and upload the archive there, then re-run this cell.")
    mste_info = {"exists": False}
else:
    fname = archive_candidates[0]
    fpath = os.path.join(DRIVE_MSTE_INCOMING, fname)
    size = os.path.getsize(fpath)
    print(f"Found MST-E archive: {fname} ({size/1e6:.1f} MB) -- left as-is (not extracted).")
    mste_info = {"exists": True, "archive_filename": fname, "size_mb": round(size / 1e6, 1)}


No archive found yet in: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/mst-e/_incoming
Download from https://skintone.google/mste-dataset and upload the archive there, then re-run this cell.


## Step 6 — Force a full Drive sync, then re-verify EVERYTHING before trusting it

This is the step the original notebook was missing. `flush_and_unmount()` forces Colab to wait for all
pending Drive writes to actually complete, rather than assuming they already have. We then remount and
re-check every archive's size from a **fresh mount** — the only way to be sure this session's writes didn't
just get lost the same way the first run's did.

In [ ]:
drive.flush_and_unmount()
print("Drive unmounted -- forced full sync of any pending writes.")

import time
time.sleep(3)
drive.mount('/content/drive')
print("Drive remounted.")


Drive unmounted -- forced full sync of any pending writes.
Mounted at /content/drive
Drive remounted.


In [ ]:
# Re-verify from the fresh mount -- do not trust anything printed before this cell.
expected = {
    "fairface.tar.gz": (DRIVE_FAIRFACE_DIR, fairface_drive_size),
    "fitzpatrick17k.tar.gz": (DRIVE_FITZ_DIR, fitz_drive_size),
}

all_ok = True
for archive_name, (drive_dir, expected_size) in expected.items():
    path = os.path.join(drive_dir, archive_name)
    if not os.path.exists(path):
        print(f"❌ MISSING after remount: {path}")
        all_ok = False
        continue
    actual_size = os.path.getsize(path)
    if actual_size != expected_size:
        print(f"❌ SIZE MISMATCH after remount: {path} -- expected {expected_size}, found {actual_size}")
        all_ok = False
    else:
        print(f"✅ Confirmed after fresh remount: {archive_name} ({actual_size/1e6:.1f} MB)")

if all_ok:
    print("\n✅✅ All archives verified present and correctly sized on Drive after a full remount.")
    print("This is a much stronger guarantee than the original notebook had -- safe to close this session.")
else:
    print("\n⚠️  Something did not verify. Do NOT close this session yet -- re-run the affected archive step.")


✅ Confirmed after fresh remount: fairface.tar.gz (2964.1 MB)
✅ Confirmed after fresh remount: fitzpatrick17k.tar.gz (233.1 MB)

✅✅ All archives verified present and correctly sized on Drive after a full remount.
This is a much stronger guarantee than the original notebook had -- safe to close this session.


## Step 7 — Write the manifest (last, and only after verification passed)

The manifest is a single small file, so it isn't itself at risk the way thousands of images were — but we
still write it last, deliberately, so its mere presence is a signal that everything before it in this
notebook completed and verified successfully.

In [ ]:
import json
from datetime import datetime, timezone

manifest = {
    "layer": "01_skin_tone_color_classification",
    "task": "L1_T1_data_gathering",
    "version": "v2_archive_based",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "storage_pattern": (
        "Each dataset is stored on Drive as ONE .tar.gz archive, not individual files. "
        "This avoids the Google Drive FUSE-mount async-sync data loss that affected the v1 notebook. "
        "L1_T2 and later notebooks must extract these archives to local disk before use."
    ),
    "datasets": {
        "fairface": {
            "drive_archive": os.path.join(DRIVE_FAIRFACE_DIR, "fairface.tar.gz"),
            "archive_size_mb": round(fairface_drive_size / 1e6, 1),
        },
        "fitzpatrick17k": {
            "drive_archive": os.path.join(DRIVE_FITZ_DIR, "fitzpatrick17k.tar.gz"),
            "archive_size_mb": round(fitz_drive_size / 1e6, 1),
            "usable_image_count": len(usable_df),
            "class_balance_note": (
                "dermaamin.com is permanently offline (NXDOMAIN, confirmed manually) -- ~76% of the raw "
                "16,577 URLs are unreachable and will stay that way. Surviving images skew MORE toward "
                "Fitzpatrick IV-VI than the original dataset did."
            ),
            "class_distribution": usable_df["fitzpatrick_scale"].value_counts().sort_index().to_dict(),
        },
        "mst_e": mste_info,
    },
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("\nManifest saved to:", MANIFEST_PATH)


{
  "layer": "01_skin_tone_color_classification",
  "task": "L1_T1_data_gathering",
  "version": "v2_archive_based",
  "generated_at_utc": "2026-09-27T00:24:32.378864+00:00",
  "storage_pattern": "Each dataset is stored on Drive as ONE .tar.gz archive, not individual files. This avoids the Google Drive FUSE-mount async-sync data loss that affected the v1 notebook. L1_T2 and later notebooks must extract these archives to local disk before use.",
  "datasets": {
    "fairface": {
      "drive_archive": "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/fairface/fairface.tar.gz",
      "archive_size_mb": 2964.1
    },
    "fitzpatrick17k": {
      "drive_archive": "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/fitzpatrick17k/fitzpatrick17k.tar.gz",
      "archive_size_mb": 233.1,
      "usable_image_count": 3734,
      "class_balance_note": "dermaamin.com is permanently offline (NXDOMAIN, confirmed manually) -- ~76% of the raw 16,577 URLs are unreachable and wi

## Done — what to do next

- Everything durable now lives on Drive as **archives**, not loose files:
  `Projects/Aura/Layer1_SkinTone/datasets/fairface/fairface.tar.gz`,
  `.../fitzpatrick17k/fitzpatrick17k.tar.gz`, and (once you've done the manual step)
  `.../mst-e/_incoming/<whatever you uploaded>`.
- **Do not** manually extract these onto Drive — always extract to Colab's local disk in whichever notebook
  needs the individual files, exactly the way `L1_T2` will.
- The old `Ideas/AURA/`, `Ideas/Aura/`, and `Aura_ML/` folders from earlier attempts are now stale duplicates.
  Once you've confirmed this run's archives are good (Step 6 said all-clear), it's safe to delete those three
  folders from Drive to reclaim space.
- Next notebook: **`L1_T2_Preprocessing`** — will be updated to extract `fairface.tar.gz` and
  `fitzpatrick17k.tar.gz` to local disk first, then run face detection / cropping / lighting normalization
  from there, exactly like this notebook now does for downloading.


## Task Summary — L1_T1: Data Gathering (v2, archive-based)

**Goal:** Gather and durably store the three datasets identified for Layer 1 (Skin Tone / Color
Classification) — FairFace, Fitzpatrick17k, and MST-E — so later notebooks can load them without
re-downloading, and without the data-loss risk that affected the first version of this notebook.

**Process:**
1. Diagnosed the root cause of the first run's data loss: Google Drive's Colab mount syncs writes to the
   cloud asynchronously, and writing ~100,000 individual small files is a worst-case pattern for that —
   session end can silently lose anything not yet synced, even though the notebook reports success.
2. Rebuilt the pipeline to write everything to Colab's local disk first, archive each dataset into a single
   `.tar.gz`, copy just that one file to Drive, and verify its size matches before trusting it.
3. Re-downloaded FairFace (Hugging Face `datasets` library) and Fitzpatrick17k (GitHub CSV + per-image
   scrape) using this new pattern.
4. Forced a full Drive sync (`flush_and_unmount` + remount) and re-verified every archive from a fresh mount
   — the check the first version never had — before writing the manifest.

**Accomplished:**
- **FairFace and Fitzpatrick17k are now stored as verified single-file archives on Drive**, confirmed present
  and correctly sized after an explicit unmount/remount cycle — not just trusted from in-session print
  statements the way the first attempt was.
- Fitzpatrick17k's class-balance finding from the first attempt holds: the ~3,700 images that survive
  `dermaamin.com`'s permanent outage are better balanced toward Fitzpatrick IV–VI than the original dataset.
- A `manifest.json` documenting the new archive-based storage pattern, so every future notebook in this
  series knows to extract-then-use rather than read directly from Drive.

**What was not included, and why:**
- **MST-E** — still a manual download (Google provides no API), but now handled safely: you upload the
  archive itself directly, and this notebook deliberately does not extract it onto Drive, to avoid recreating
  the exact failure mode this whole rebuild was meant to fix.
- **The ~76% of Fitzpatrick17k tied to `dermaamin.com`** — unchanged from the first attempt: permanently
  unreachable, confirmed via direct DNS test, not fixable by any retry strategy.
- **Extraction/preprocessing of either archive** — deliberately left to `L1_T2`. This notebook's job ends at
  "safely and verifiably stored," not "ready to train on."


In [ ]:
import tarfile

def inspect_archive(drive_path, label):
    print(f"--- {label} ---")
    print(f"Path: {drive_path}")
    print(f"Size on Drive: {os.path.getsize(drive_path)/1e6:.1f} MB")
    with tarfile.open(drive_path, "r:gz") as tar:
        members = tar.getmembers()
        jpgs = [m for m in members if m.name.endswith(".jpg")]
        csvs = [m for m in members if m.name.endswith(".csv")]
        print(f"Total members: {len(members)} | .jpg files: {len(jpgs)} | .csv files: {len(csvs)}")
        # Try reading the last file in the archive fully -- if the archive got cut off
        # mid-write, this is where it would fail
        last_member = members[-1]
        f = tar.extractfile(last_member)
        if f:
            data = f.read()
            print(f"Last member ({last_member.name}) read OK: {len(data)} bytes")
    print()

inspect_archive(
    "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/fairface/fairface.tar.gz",
    "FairFace"
)
inspect_archive(
    "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/fitzpatrick17k/fitzpatrick17k.tar.gz",
    "Fitzpatrick17k"
)

--- FairFace ---
Path: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/fairface/fairface.tar.gz
Size on Drive: 2964.1 MB
Total members: 97703 | .jpg files: 97698 | .csv files: 2
Last member (fairface/validation_labels.csv) read OK: 197198 bytes

--- Fitzpatrick17k ---
Path: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/datasets/fitzpatrick17k/fitzpatrick17k.tar.gz
Size on Drive: 233.1 MB
Total members: 3892 | .jpg files: 3887 | .csv files: 3
Last member (fitzpatrick17k/images/016576.jpg) read OK: 56883 bytes

